In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Setup & models:

In [2]:
!pip install -q faiss-cpu

import torch, pandas as pd, numpy as np, faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import pipeline
from tqdm.auto import tqdm

DATA    = "/kaggle/input/competitions/smart-mcq-solver-challenge"
OPTIONS = ["A", "B", "C", "D", "E"]
DEV     = 0 if torch.cuda.is_available() else -1
TOP_K   = 5

train = pd.read_csv(f"{DATA}/train.csv")
test  = pd.read_csv(f"{DATA}/test.csv")

kb = [str(row[row["answer"]]) for _, row in train.iterrows()]

embedder = SentenceTransformer("all-MiniLM-L6-v2")
kb_emb = embedder.encode(kb, show_progress_bar=True).astype("float32")
index = faiss.IndexFlatL2(kb_emb.shape[1])
index.add(kb_emb)

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=DEV)
print("Pipeline ready. KB docs:", index.ntotal)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 72.0 MB/s eta 0:00:00:00:0100:01


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Pipeline ready. KB docs: 2000


# RAG function

In [3]:
def retrieve_indices(prompt, k, exclude=None):
    q = embedder.encode([str(prompt)]).astype("float32")
    _, I = index.search(q, k + (1 if exclude is not None else 0))
    idxs = [int(i) for i in I[0] if i != exclude][:k]
    return idxs

def rag_top3(prompt, options_texts, exclude=None):
    docs = [kb[i] for i in retrieve_indices(prompt, TOP_K, exclude)]
    ce = cross_encoder.predict([[str(prompt), d] for d in docs])
    best = docs[int(np.argmax(ce))]
    rag = f"Context: {best} Question: {prompt}"
    res = zs(rag, candidate_labels=options_texts)
    ranked = [OPTIONS[options_texts.index(lab)] for lab in res["labels"]]
    return ranked

def map3_row(ranked, truth):
    for i, l in enumerate(ranked[:3]):
        if l == truth:
            return 1.0 / (i + 1)
    return 0.0

# Honest train eval

In [4]:
N_EVAL = 100
maps = []
for i in tqdm(range(N_EVAL), desc="train LOO eval"):
    r = train.iloc[i]
    opts = [str(r[o]) for o in OPTIONS]
    ranked = rag_top3(str(r["prompt"]), opts, exclude=i)
    maps.append(map3_row(ranked, r["answer"]))
train_map3 = float(np.mean(maps))
print("Train LOO MAP@3 (100 rows):", round(train_map3, 4))

train LOO eval:   0%|          | 0/100 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Train LOO MAP@3 (100 rows): 0.9117


# Test submission

In [5]:
preds = []
for i in tqdm(range(len(test)), desc="test submission"):
    r = test.iloc[i]
    opts = [str(r[o]) for o in OPTIONS]
    ranked = rag_top3(str(r["prompt"]), opts)     
    preds.append(" ".join(ranked[:3]))

submission = pd.DataFrame({"ID": test["id"], "Prediction": preds})
submission.to_csv("submission.csv", index=False)
print("submission.csv saved:", submission.shape)
submission.head()

test submission:   0%|          | 0/500 [00:00<?, ?it/s]

submission.csv saved: (500, 2)


,ID,Prediction
0,1,A D E
1,2,E D B
2,3,B D E
3,4,E A D
4,5,C A D


# W&B logging

In [6]:
import wandb
from kaggle_secrets import UserSecretsClient
wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))

run = wandb.init(entity="ishankgpt02-na", project="23f1002033-t22026",
                 name="rag-faiss-crossencoder",
                 config={"retriever": "all-MiniLM-L6-v2",
                         "reranker": "ms-marco-MiniLM-L-6-v2",
                         "reader": "bart-large-mnli (zero-shot)",
                         "top_k": TOP_K})
wandb.log({"train_loo_map@3": train_map3})
run.summary["note"] = ("RAG: FAISS retrieve -> cross-encoder rerank -> zero-shot read. "
                       "Bottleneck is the zero-shot reader (~0.45), so RAG doesn't beat "
                       "the supervised TF-IDF+LR (0.74) on this artifact-driven dataset.")
run.finish()
print("Logged to W&B")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ishankgpt02 (ishankgpt02-na) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


train_loo_map@3,▁
note,RAG: FAISS retrieve ...
train_loo_map@3,0.91167


Logged to W&B
